In [1]:
import pandas as pd

### First step is creating some dummy data to repliacte the post and login dataframes. `big_date_df` has values for every two hours while `small_date_df` has values for every ten hours (less frequent).
Each column is named `datetime` and each cell holds datetime values.

This approach works if there are exact matches, but we might want to also use the `.floor()` method to simplify the post and login values to remove the miliseconds or remove the seconds.

In [2]:
big_date_df = pd.DataFrame({'datetime': pd.date_range(start = pd.to_datetime('2026-01-30, 00:01:59.30'), end = pd.to_datetime('2026-12-31, 00:01:59.30'), freq = '2h')})
small_date_df = pd.DataFrame({'datetime': pd.date_range(start = pd.to_datetime('2026-01-30, 00:01:59.30'), end = pd.to_datetime('2026-12-31, 00:01:59.30'), freq = '10h')})

In [3]:
big_date_df.head()

,datetime
0,2026-01-30 00:01:59.300
1,2026-01-30 02:01:59.300
2,2026-01-30 04:01:59.300
3,2026-01-30 06:01:59.300
4,2026-01-30 08:01:59.300


In [4]:
type(big_date_df['datetime'].iloc[0]), big_date_df.shape

(pandas._libs.tslibs.timestamps.Timestamp, (4021, 1))

In [5]:
small_date_df.head()

,datetime
0,2026-01-30 00:01:59.300
1,2026-01-30 10:01:59.300
2,2026-01-30 20:01:59.300
3,2026-01-31 06:01:59.300
4,2026-01-31 16:01:59.300


In [6]:
type(small_date_df['datetime'].iloc[0]), small_date_df.shape

(pandas._libs.tslibs.timestamps.Timestamp, (805, 1))

### Below is the example of the floor method.
The first creates a floor at the seconds level (removing milliseconds). The second creates a floor at the minute value (removing milliseconds and seconds).

Use this for both the posts and login datetimes if the filtering does not return any results.

In [7]:
big_date_df['datetime'].dt.floor('s').head()

0   2026-01-30 00:01:59
1   2026-01-30 02:01:59
2   2026-01-30 04:01:59
3   2026-01-30 06:01:59
4   2026-01-30 08:01:59
Name: datetime, dtype: datetime64[ns]

In [8]:
big_date_df['datetime'].dt.floor('min').head()

0   2026-01-30 00:01:00
1   2026-01-30 02:01:00
2   2026-01-30 04:01:00
3   2026-01-30 06:01:00
4   2026-01-30 08:01:00
Name: datetime, dtype: datetime64[ns]

### Below is the logic for filtering.
Instead of using `in` which is the general use for these kinds of conditional statements we are using the pandas builtin version `.isin()`. The logic is the same.

In [9]:
big_date_df['datetime'].iloc[0:10] == small_date_df['datetime'].iloc[0]

0     True
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: datetime, dtype: bool

In [10]:
big_date_df[big_date_df['datetime'] == small_date_df['datetime'].iloc[0]]

,datetime
0,2026-01-30 00:01:59.300


In [11]:
filtered_date_df = big_date_df[big_date_df['datetime'].isin(small_date_df['datetime'])]
filtered_date_df.head()

,datetime
0,2026-01-30 00:01:59.300
5,2026-01-30 10:01:59.300
10,2026-01-30 20:01:59.300
15,2026-01-31 06:01:59.300
20,2026-01-31 16:01:59.300


In [12]:
filtered_date_df.shape

(805, 1)

In [13]:
filtered_date_df.index[0:10]

Index([0, 5, 10, 15, 20, 25, 30, 35, 40, 45], dtype='int64')

In [14]:
filtered_date_df.index[0:10] - 1

Index([-1, 4, 9, 14, 19, 24, 29, 34, 39, 44], dtype='int64')

In [15]:
last_dates_removed = big_date_df.drop(axis = 0, labels = filtered_date_df.index[1:] - 1)
last_dates_removed.head(20)

,datetime
0,2026-01-30 00:01:59.300
1,2026-01-30 02:01:59.300
2,2026-01-30 04:01:59.300
3,2026-01-30 06:01:59.300
5,2026-01-30 10:01:59.300
6,2026-01-30 12:01:59.300
7,2026-01-30 14:01:59.300
8,2026-01-30 16:01:59.300
10,2026-01-30 20:01:59.300
11,2026-01-30 22:01:59.300


### And all the examples from above demonstrating this can be reduced down to a more general function that accepts the post dataframe and the login dataframe to return a new dataframe that removes the last post from each day.
Remember that this is done AFTER calculating the time deltas for each post.

In [16]:
def remove_last_post(post_df, login_df):
    filtered_df = post_df[post_df['datetime'].isin(login_df['datetime'])]
    new_df = post_df.drop(axis = 0, labels = filtered_df.index[1:] - 1)
    return new_df

In [17]:
last_post_removed_df = remove_last_post(big_date_df, small_date_df)
last_post_removed_df.head(20)

,datetime
0,2026-01-30 00:01:59.300
1,2026-01-30 02:01:59.300
2,2026-01-30 04:01:59.300
3,2026-01-30 06:01:59.300
5,2026-01-30 10:01:59.300
6,2026-01-30 12:01:59.300
7,2026-01-30 14:01:59.300
8,2026-01-30 16:01:59.300
10,2026-01-30 20:01:59.300
11,2026-01-30 22:01:59.300
